In [ ]:
!pip uninstall -y bitsandbytes triton
!pip install --no-cache-dir --upgrade "bitsandbytes>=0.45.5" "accelerate" "transformers" "peft" "trl"

Found existing installation: triton 3.6.0
Uninstalling triton-3.6.0:
  Successfully uninstalled triton-3.6.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 213.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 133.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 223.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 249.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 102.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attemp

In [ ]:
import torch
import bitsandbytes as bnb

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("bitsandbytes:", bnb.__version__)

CUDA available: True
GPU: Tesla T4
Torch: 2.10.0+cu128
CUDA: 12.8
bitsandbytes: 0.49.2


In [ ]:
import torch, triton, bitsandbytes, transformers, trl, peft
print("torch:", torch.__version__)
print("triton:", triton.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")


torch: 2.10.0+cu128
triton: 3.6.0
bitsandbytes: 0.49.2
transformers: 5.5.4
trl: 1.2.0
peft: 0.19.1
cuda available: True
gpu: Tesla T4


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:


DRIVE_DIR = "/content/drive/MyDrive/CS_F425_Project"  # change if needed

# Use True only once if you want to ignore all previous saved artifacts and start clean.
# After the first successful run, set this to False so Colab disconnects can resume.
FORCE_FRESH_TRAIN = False

import ast
import gc
import glob as _glob
import inspect as _inspect
import json
import os
import random
import re
import sys
import time as _time
from numbers import Integral, Real
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import torch
from datasets import Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    StoppingCriteria,
    StoppingCriteriaList,
    TrainerCallback,
)

sys.path.insert(0, DRIVE_DIR)
from tool_executor import ToolExecutor

MODEL_NAME_P2 = "mistralai/Mistral-7B-v0.1"
ADAPTER_DIR_P2 = f"{DRIVE_DIR}/mistral-react-adapter_divyam_duplicate"
CKPT_DIR_P2 = f"{DRIVE_DIR}/mistral-react-qlora_divyam_duplicate"
MAX_SEQ_LENGTH_P2 = 384

os.makedirs(ADAPTER_DIR_P2, exist_ok=True)
os.makedirs(CKPT_DIR_P2, exist_ok=True)

df = pd.read_csv(f"{DRIVE_DIR}/sales_data.csv")
with open(f"{DRIVE_DIR}/agent_trajectories_2k.json", "r", encoding="utf-8") as f:
    trajectories = json.load(f)

print("data shape:", df.shape)
print("trajectories:", len(trajectories))


def detect_p2_state():
    if os.path.isfile(f"{ADAPTER_DIR_P2}/adapter_config.json"):
        return "TRAINING_COMPLETE", ADAPTER_DIR_P2

    epoch_ckpts = _glob.glob(f"{CKPT_DIR_P2}/checkpoint-*")
    if epoch_ckpts:
        epoch_ckpts.sort(key=lambda p: int(p.rsplit("-", 1)[-1]))
        return "EPOCH_CHECKPOINT", epoch_ckpts[-1]

    timed_ckpts = _glob.glob(f"{ADAPTER_DIR_P2}/timed_ckpt_step_*")
    if timed_ckpts:
        timed_ckpts.sort(key=lambda p: int(p.rsplit("_", 1)[-1]))
        return "TIMED_CHECKPOINT", timed_ckpts[-1]

    return "FRESH", None


def _parse_numeric(val_str):
    return float(val_str) if "." in val_str else int(val_str)


def parse_agent_action(action_str):
    if action_str.startswith("filter_data"):
        m = re.search(r"column='([^']+)',\s*value=(-?[\d]+(?:\.\d+)?)", action_str)
        if m:
            return {
                "tool": "filter",
                "args": {"column": m.group(1), "op": "==", "value": _parse_numeric(m.group(2))},
            }
        m = re.search(r"column='([^']+)',\s*value='([^']+)'", action_str)
        if m:
            return {
                "tool": "filter",
                "args": {"column": m.group(1), "op": "==", "value": m.group(2)},
            }

    elif action_str.startswith("group_by"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "groupby", "args": {"column": m.group(1)}}

    elif action_str.startswith("aggregate_sum"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "sum"}}

    elif action_str.startswith("aggregate_mean"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "mean"}}

    elif action_str.startswith("aggregate_count"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "count"}}

    elif action_str.startswith("sort_by"):
        m = re.search(r"column='([^']+)',\s*order='([^']+)'", action_str)
        if m:
            return {
                "tool": "sort",
                "args": {"column": m.group(1), "ascending": m.group(2) != "desc"},
            }

    elif action_str.startswith("top_k"):
        m = re.search(r"k=(\d+)", action_str)
        if m:
            return {"tool": "topk", "args": {"k": int(m.group(1))}}

    return None


def clean_scalar(value):
    if hasattr(value, "item"):
        try:
            value = value.item()
        except Exception:
            pass

    if isinstance(value, bool):
        return bool(value)
    if isinstance(value, Integral):
        return int(value)
    if isinstance(value, Real):
        return round(float(value), 4)
    return value


def result_to_python(actions, result_df):
    if result_df is None or (hasattr(result_df, "empty") and result_df.empty):
        return None
    if hasattr(result_df, "groups"):
        return None

    action_names = [a.split("(", 1)[0] for a in actions]

    if getattr(result_df, "shape", None) == (1, 1):
        return clean_scalar(result_df.iloc[0, 0])

    has_groupby = any(a.startswith("group_by") for a in action_names)
    has_sort_or_topk = any(
        a.startswith("sort_by") or a.startswith("top_k") for a in action_names
    )

    if isinstance(result_df, pd.DataFrame):
        if result_df.shape[1] == 2 and has_groupby and not has_sort_or_topk:
            key_col, value_col = result_df.columns
            return {
                str(row[key_col]): clean_scalar(row[value_col])
                for _, row in result_df.iterrows()
            }

        return [
            {str(key): clean_scalar(value) for key, value in row.items()}
            for row in result_df.to_dict(orient="records")
        ]

    return clean_scalar(result_df)


def execute_action_sequence(actions, source_df):
    parsed = []
    for action in actions:
        parsed_action = parse_agent_action(action)
        if parsed_action is None:
            raise ValueError(f"Could not parse action: {action}")
        parsed.append(parsed_action)
    return ToolExecutor(source_df.copy()).execute(parsed)


def compute_answer(actions, source_df):
    if not actions:
        return None
    try:
        result = execute_action_sequence(actions, source_df)
        return result_to_python(actions, result)
    except Exception:
        return None


REACT_SYSTEM_PROMPT = """You are a data analysis agent. You have access to the following tools to analyze a sales dataset:

Function Descriptions:
[
  {"name": "filter_data", "description": "Filter rows where column equals value", "parameters": {"column": "str", "value": "str or int"}},
  {"name": "group_by", "description": "Group the data by a column", "parameters": {"column": "str"}},
  {"name": "aggregate_sum", "description": "Sum a numeric column", "parameters": {"column": "str"}},
  {"name": "aggregate_mean", "description": "Average a numeric column", "parameters": {"column": "str"}},
  {"name": "aggregate_count", "description": "Count rows for a column", "parameters": {"column": "str"}},
  {"name": "sort_by", "description": "Sort by a column", "parameters": {"column": "str", "order": "asc or desc"}},
  {"name": "top_k", "description": "Select top k rows", "parameters": {"k": "int"}}
]

Schema: date (date), year (int), month (int), city (str), region (str), product (str), category (str), revenue (float), units_sold (int), cost (float), profit (float)

Use the Thought/Action/Action Input format. Wait for Observation after each action. End with Final Answer."""

REACT_PROMPT_TEMPLATE = """### System
{system}

### User Query
{question}

### Agent Scratchpad
{scratchpad}"""

_THOUGHT_TEMPLATES = {
    "filter_data": "I need to filter the data by {args} to narrow down the dataset.",
    "group_by": "I should group the data by {args} to organize it.",
    "aggregate_sum": "I need to compute the sum of {args}.",
    "aggregate_mean": "I need to compute the average of {args}.",
    "aggregate_count": "I need to count the entries for {args}.",
    "sort_by": "I should sort the results by {args}.",
    "top_k": "I need to select the top {args} entries.",
}


def action_string_to_action_input(action_str):
    if action_str.startswith("filter_data"):
        m = re.search(r"column='([^']+)',\s*value=(-?[\d]+(?:\.\d+)?)", action_str)
        if m:
            return {"column": m.group(1), "value": _parse_numeric(m.group(2))}
        m = re.search(r"column='([^']+)',\s*value='([^']+)'", action_str)
        if m:
            return {"column": m.group(1), "value": m.group(2)}

    elif action_str.startswith("group_by"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"column": m.group(1)}

    elif action_str.startswith(("aggregate_sum", "aggregate_mean", "aggregate_count")):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"column": m.group(1)}

    elif action_str.startswith("sort_by"):
        m = re.search(r"column='([^']+)',\s*order='([^']+)'", action_str)
        if m:
            return {"column": m.group(1), "order": m.group(2)}

    elif action_str.startswith("top_k"):
        m = re.search(r"k=(\d+)", action_str)
        if m:
            return {"k": int(m.group(1))}

    raise ValueError(f"Unsupported action string: {action_str}")


def _quote_string(value):
    escaped = value.replace("\\", "\\\\").replace("'", "\\'")
    return f"'{escaped}'"


def _format_dsl_value(value):
    if isinstance(value, str):
        return _quote_string(value)
    if isinstance(value, bool):
        return "1" if value else "0"
    return str(value)


def action_input_to_action_string(action_name, action_input):
    if action_name == "filter_data":
        return (
            f"filter_data(column={_quote_string(str(action_input['column']))}, "
            f"value={_format_dsl_value(action_input['value'])})"
        )
    if action_name == "group_by":
        return f"group_by(column={_quote_string(str(action_input['column']))})"
    if action_name in {"aggregate_sum", "aggregate_mean", "aggregate_count"}:
        return f"{action_name}(column={_quote_string(str(action_input['column']))})"
    if action_name == "sort_by":
        return (
            f"sort_by(column={_quote_string(str(action_input['column']))}, "
            f"order={_quote_string(str(action_input['order']))})"
        )
    if action_name == "top_k":
        return f"top_k(k={int(action_input['k'])})"
    raise ValueError(f"Unknown action name: {action_name}")


def format_observation(result_df, max_chars=300):
    if result_df is None:
        return "No data returned."
    if hasattr(result_df, "groups") and not isinstance(result_df, pd.DataFrame):
        group_names = list(result_df.groups.keys())
        return f"Grouped result with {len(group_names)} groups. Sample groups: {[str(x) for x in group_names[:10]]}"
    if hasattr(result_df, "empty") and result_df.empty:
        return "Empty result."
    if getattr(result_df, "shape", None) == (1, 1):
        return str(result_df.iloc[0, 0])

    text = result_df.to_string(index=False) if isinstance(result_df, pd.DataFrame) else str(result_df)
    if len(text) > max_chars:
        text = text[:max_chars] + "..."
    return text


def build_react_trace(entry, source_df, max_obs_chars=300):
    actions = entry["actions"]
    trace_lines = []
    executed_actions = []

    for idx, action_str in enumerate(actions):
        action_name = action_str.split("(", 1)[0]
        action_input = action_string_to_action_input(action_str)
        args_text = json.dumps(action_input, ensure_ascii=False)

        thought = _THOUGHT_TEMPLATES.get(
            action_name,
            "I need to use the next tool with {args}.",
        ).format(args=args_text)

        if idx == 0:
            thought = "Let me solve this step by step. " + thought

        trace_lines.append(f"Thought: {thought}")
        trace_lines.append(f"Action: {action_name}")
        trace_lines.append(f"Action Input: {args_text}")

        executed_actions.append(action_str)

        try:
            result = execute_action_sequence(executed_actions, source_df)
        except Exception:
            return None

        trace_lines.append(f"Observation: {format_observation(result, max_chars=max_obs_chars)}")

    final_answer = compute_answer(actions, source_df)
    if final_answer is None:
        return None

    trace_lines.append("Thought: I now have all the information needed.")
    trace_lines.append(f"Final Answer: {json.dumps(final_answer, ensure_ascii=False)}")
    return "\n".join(trace_lines)


def build_react_training_data(trajectories, source_df, max_obs_chars=300):
    react_data = []
    skipped = 0

    for entry in tqdm(trajectories, desc="Building ReAct traces"):
        trace = build_react_trace(entry, source_df, max_obs_chars=max_obs_chars)
        if trace is None:
            skipped += 1
            continue
        react_data.append({"question": entry["query"], "trace": trace})

    print(f"ReAct traces built: {len(react_data)} / {len(trajectories)}")
    print(f"Skipped           : {skipped}")
    if react_data:
        print("\nSample trace:\n")
        print(react_data[0]["trace"][:1200])
    return react_data


def format_react_example(row, eos_token):
    prompt = REACT_PROMPT_TEMPLATE.format(
        system=REACT_SYSTEM_PROMPT,
        question=row["question"],
        scratchpad="",
    )
    return {"text": prompt + row["trace"] + eos_token}


def load_tokenizer_p2():
    tokenizer_p2 = AutoTokenizer.from_pretrained(MODEL_NAME_P2, trust_remote_code=True)
    if tokenizer_p2.pad_token is None:
        tokenizer_p2.pad_token = tokenizer_p2.eos_token
    tokenizer_p2.padding_side = "right"
    return tokenizer_p2


react_data = build_react_training_data(trajectories, df)
tokenizer_p2 = load_tokenizer_p2()
EOS_P2 = tokenizer_p2.eos_token

random.seed(42)
random.shuffle(react_data)

train_size = min(1800, int(len(react_data) * 0.9))
valid_size = min(200, len(react_data) - train_size)

raw_train = react_data[:train_size]
raw_valid = react_data[train_size : train_size + valid_size]

def fmt(row):
    return format_react_example(row, eos_token=EOS_P2)

train_ds = Dataset.from_list(raw_train).map(fmt, remove_columns=["question", "trace"])
valid_ds = Dataset.from_list(raw_valid).map(fmt, remove_columns=["question", "trace"])

print("phase2 train:", len(train_ds))
print("phase2 valid:", len(valid_ds))
print("sample text:", train_ds[0]["text"][:500])

P2_STATE, P2_CKPT_PATH = detect_p2_state()

if FORCE_FRESH_TRAIN:
    P2_SKIP_TRAINING = False
    P2_CKPT_PATH = None
else:
    P2_SKIP_TRAINING = P2_STATE in ("TRAINING_COMPLETE", "TIMED_CHECKPOINT")

print("Phase 2 state:", P2_STATE)
print("Checkpoint:", P2_CKPT_PATH)
print("Skip training:", P2_SKIP_TRAINING)
print("Epoch checkpoints dir:", CKPT_DIR_P2)


data shape: (10000, 11)
trajectories: 2000


Building ReAct traces:   0%|          | 0/2000 [00:00<?, ?it/s]

ReAct traces built: 2000 / 2000
Skipped           : 0

Sample trace:

Thought: Let me solve this step by step. I need to filter the data by {"column": "year", "value": 2021} to narrow down the dataset.
Action: filter_data
Action Input: {"column": "year", "value": 2021}
Observation:       date  year  month      city region product    category  revenue  units_sold     cost   profit
2021-09-11  2021      9     Delhi  North       B    Clothing 10506.89          65  7231.71  3275.19
2021-03-20  2021      3   Kolkata   East       B    Clothing 10113.05          64  7138.90  2974.15
...
Thought: I need to compute the sum of {"column": "revenue"}.
Action: aggregate_sum
Action Input: {"column": "revenue"}
Observation: 44882702.019999996
Thought: I now have all the information needed.
Final Answer: 44882702.02


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

phase2 train: 1800
phase2 valid: 200
sample text: ### System
You are a data analysis agent. You have access to the following tools to analyze a sales dataset:

Function Descriptions:
[
  {"name": "filter_data", "description": "Filter rows where column equals value", "parameters": {"column": "str", "value": "str or int"}},
  {"name": "group_by", "description": "Group the data by a column", "parameters": {"column": "str"}},
  {"name": "aggregate_sum", "description": "Sum a numeric column", "parameters": {"column": "str"}},
  {"name": "aggregate_m
Phase 2 state: FRESH
Checkpoint: None
Skip training: False
Epoch checkpoints dir: /content/drive/MyDrive/CS_F425_Project/mistral-react-qlora_divyam_duplicate


In [ ]:
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

# Reduce the chance of fragmentation after reconnects / failed runs
gc.collect()
torch.cuda.empty_cache()

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config_p2 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

resume_ckpt = None
timed_resume_ckpt = None

if not FORCE_FRESH_TRAIN:
    ckpts = _glob.glob(f"{CKPT_DIR_P2}/checkpoint-*")
    if ckpts:
        ckpts.sort(key=lambda p: int(p.rsplit("-", 1)[-1]))
        resume_ckpt = ckpts[-1]
    else:
        timed_ckpts = _glob.glob(f"{ADAPTER_DIR_P2}/timed_ckpt_step_*")
        if timed_ckpts:
            timed_ckpts.sort(key=lambda p: int(p.rsplit("_", 1)[-1]))
            timed_resume_ckpt = timed_ckpts[-1]

if P2_SKIP_TRAINING and P2_CKPT_PATH is not None:
    print("Loading saved Phase 2 adapter from:", P2_CKPT_PATH)
    base_model_p2 = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME_P2,
        quantization_config=bnb_config_p2,
        device_map="auto",
        trust_remote_code=True,
    )
    model_p2 = PeftModel.from_pretrained(base_model_p2, P2_CKPT_PATH)
    model_p2.eval()
    model_p2.config.use_cache = True

else:
    print("Preparing model for Phase 2 training")
    if resume_ckpt is not None:
        print("Will resume full trainer state from:", resume_ckpt)
    elif timed_resume_ckpt is not None:
        print("No trainer checkpoint found. Continuing from timed adapter:", timed_resume_ckpt)
    else:
        print("No saved checkpoints found. Starting fresh.")

    base_model_p2 = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME_P2,
        quantization_config=bnb_config_p2,
        device_map="auto",
        trust_remote_code=True,
    )
    base_model_p2.config.use_cache = False
    base_model_p2 = prepare_model_for_kbit_training(base_model_p2, use_gradient_checkpointing=True)

    if timed_resume_ckpt is not None:
        model_p2 = PeftModel.from_pretrained(base_model_p2, timed_resume_ckpt, is_trainable=True)
    else:
        lora_config_p2 = LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        )
        model_p2 = get_peft_model(base_model_p2, lora_config_p2)

    model_p2.print_trainable_parameters()

    sft_sig = set(_inspect.signature(SFTConfig.__init__).parameters.keys())
    trainer_sig = set(_inspect.signature(SFTTrainer.__init__).parameters.keys())

    eval_key = "eval_strategy" if "eval_strategy" in sft_sig else "evaluation_strategy"
    tok_key = "processing_class" if "processing_class" in trainer_sig else "tokenizer"

    cfg_extra = {}
    trainer_extra = {}
    for param, value in [
        ("max_seq_length", MAX_SEQ_LENGTH_P2),
        ("dataset_text_field", "text"),
        ("packing", False),
    ]:
        if param in sft_sig:
            cfg_extra[param] = value
        elif param in trainer_sig:
            trainer_extra[param] = value

    # Colab-safe defaults: keeps memory down and usually finishes within a 5h window on T4/L4.
    batch_size = 1
    grad_accum = 16
    save_eval_steps = 25
    steps_per_epoch = max(1, len(train_ds) // (batch_size * grad_accum))
    warmup_steps = max(1, int(0.05 * steps_per_epoch * 2))
    total_steps = steps_per_epoch * 2

    training_args_p2 = SFTConfig(
        output_dir=CKPT_DIR_P2,
        seed=42,
        num_train_epochs=2,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=warmup_steps,
        optim="paged_adamw_8bit",
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_strategy="steps",
        save_steps=save_eval_steps,
        save_total_limit=4,
        eval_steps=save_eval_steps,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="none",
        **{eval_key: "steps"},
        **cfg_extra,
    )

    class TimedCheckpointCallbackP2(TrainerCallback):
        def __init__(self, adapter_dir, tokenizer, interval_min=5):
            self.adapter_dir = adapter_dir
            self.tokenizer = tokenizer
            self.interval_sec = interval_min * 60
            self.last_save = _time.time()

        def on_step_end(self, args, state, control, model=None, **kwargs):
            elapsed = _time.time() - self.last_save
            if elapsed >= self.interval_sec and model is not None:
                save_path = f"{self.adapter_dir}/timed_ckpt_step_{state.global_step}"
                os.makedirs(save_path, exist_ok=True)
                model.save_pretrained(save_path)
                self.tokenizer.save_pretrained(save_path)
                self.last_save = _time.time()
                print(
                    f"\n[P2 TimedCheckpoint] Saved adapter at step {state.global_step} "
                    f"-> {save_path} ({elapsed / 60:.1f} min)"
                )

    trainer_p2 = SFTTrainer(
        model=model_p2,
        **{tok_key: tokenizer_p2},
        args=training_args_p2,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        callbacks=[TimedCheckpointCallbackP2(ADAPTER_DIR_P2, tokenizer_p2, interval_min=5)],
        **trainer_extra,
    )

    print("Starting Phase 2 training")
    print("train size:", len(train_ds))
    print("valid size:", len(valid_ds))
    print("max seq len:", MAX_SEQ_LENGTH_P2)
    print("batch size:", batch_size)
    print("grad accum:", grad_accum)
    print("steps/epoch:", steps_per_epoch)
    print("total steps:", total_steps)
    print("trainer checkpoints dir:", CKPT_DIR_P2)
    print("timed adapter checkpoints dir:", ADAPTER_DIR_P2)

    trainer_p2.train(resume_from_checkpoint=resume_ckpt)

    model_p2.save_pretrained(ADAPTER_DIR_P2)
    tokenizer_p2.save_pretrained(ADAPTER_DIR_P2)
    print("saved final adapter to:", ADAPTER_DIR_P2)


Preparing model for Phase 2 training
No saved checkpoints found. Starting fresh.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting Phase 2 training
train size: 1800
valid size: 200
max seq len: 384
batch size: 1
grad accum: 16
steps/epoch: 112
total steps: 224
trainer checkpoints dir: /content/drive/MyDrive/CS_F425_Project/mistral-react-qlora_divyam_duplicate
timed adapter checkpoints dir: /content/drive/MyDrive/CS_F425_Project/mistral-react-adapter_divyam_duplicate

[P2 TimedCheckpoint] Saved adapter at step 1 -> /content/drive/MyDrive/CS_F425_Project/mistral-react-adapter_divyam_duplicate/timed_ckpt_step_1 (5.4 min)


Step,Training Loss,Validation Loss
25,0.158659,0.078114



[P2 TimedCheckpoint] Saved adapter at step 2 -> /content/drive/MyDrive/CS_F425_Project/mistral-react-adapter_divyam_duplicate/timed_ckpt_step_2 (6.3 min)

[P2 TimedCheckpoint] Saved adapter at step 3 -> /content/drive/MyDrive/CS_F425_Project/mistral-react-adapter_divyam_duplicate/timed_ckpt_step_3 (5.9 min)

[P2 TimedCheckpoint] Saved adapter at step 4 -> /content/drive/MyDrive/CS_F425_Project/mistral-react-adapter_divyam_duplicate/timed_ckpt_step_4 (6.0 min)

[P2 TimedCheckpoint] Saved adapter at step 5 -> /content/drive/MyDrive/CS_F425_Project/mistral-react-adapter_divyam_duplicate/timed_ckpt_step_5 (6.0 min)

[P2 TimedCheckpoint] Saved adapter at step 6 -> /content/drive/MyDrive/CS_F425_Project/mistral-react-adapter_divyam_duplicate/timed_ckpt_step_6 (5.9 min)

[P2 TimedCheckpoint] Saved adapter at step 7 -> /content/drive/MyDrive/CS_F425_Project/mistral-react-adapter_divyam_duplicate/timed_ckpt_step_7 (5.8 min)

[P2 TimedCheckpoint] Saved adapter at step 8 -> /content/drive/MyDriv

In [ ]:
def sanitize_json(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^```json\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    text = re.sub(r",\s*([}\]])", r"\1", text)
    return text.strip()


def parse_json_like(text: str):
    cleaned = sanitize_json(text)

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    try:
        return ast.literal_eval(cleaned)
    except (ValueError, SyntaxError):
        pass

    match = re.search(r"\{.*\}|\[.*\]", cleaned, re.DOTALL)
    if match:
        chunk = sanitize_json(match.group())
        try:
            return json.loads(chunk)
        except json.JSONDecodeError:
            pass
        try:
            return ast.literal_eval(chunk)
        except (ValueError, SyntaxError):
            pass

    raise ValueError(f"Could not parse structured input: {text}")


def parse_action_input_flexible(action_name: str, text: str):
    raw = text.strip()
    if not raw:
        raise ValueError("Empty Action Input.")

    try:
        parsed = parse_json_like(raw)
        if isinstance(parsed, dict):
            return parsed
    except Exception:
        pass

    try:
        parsed = action_string_to_action_input(f"{action_name}({raw})")
        if isinstance(parsed, dict):
            return parsed
    except Exception:
        pass

    raise ValueError(f"Could not parse Action Input for {action_name}: {raw}")


class ObservationStopCriteria(StoppingCriteria):
    def __init__(self, tokenizer, triggers=("Observation:", "Final Answer:")):
        self.tokenizer = tokenizer
        self.triggers = triggers

    def __call__(self, input_ids, scores, **kwargs):
        tail = self.tokenizer.decode(input_ids[0, -40:], skip_special_tokens=True)
        return any(trigger in tail for trigger in self.triggers)


class ReActAgent:
    def __init__(
        self,
        model,
        tokenizer,
        source_df,
        max_steps=5,
        max_obs_chars=500,
        max_consecutive_errors=2,
        max_new_tokens=256,
    ):
        self.model = model
        self.tokenizer = tokenizer
        self.source_df = source_df
        self.max_steps = max_steps
        self.max_obs_chars = max_obs_chars
        self.max_consecutive_errors = max_consecutive_errors
        self.max_new_tokens = max_new_tokens
        self.stop_criteria = StoppingCriteriaList([ObservationStopCriteria(tokenizer)])

    def run(self, question):
        self.model.eval()
        self.model.config.use_cache = True

        scratchpad = ""
        consecutive_errors = 0
        all_actions = []

        for step in range(self.max_steps):
            prompt = REACT_PROMPT_TEMPLATE.format(
                system=REACT_SYSTEM_PROMPT,
                question=question,
                scratchpad=scratchpad,
            )

            output_text = self._generate(prompt)
            scratchpad += output_text

            if "Final Answer:" in output_text:
                answer_text = output_text.split("Final Answer:", 1)[-1].strip()
                answer_text = answer_text.replace(self.tokenizer.eos_token or "", "").strip()
                return {
                    "actions": all_actions,
                    "answer": self._parse_final_answer(answer_text),
                    "steps": step + 1,
                    "scratchpad": scratchpad,
                }

            try:
                action_name, action_input = self._parse_action(output_text)
                action_str = action_input_to_action_string(action_name, action_input)

                if parse_agent_action(action_str) is None:
                    raise ValueError(f"Unknown action: {action_str}")

                all_actions.append(action_str)
                result = execute_action_sequence(all_actions, self.source_df)
                obs = format_observation(result, max_chars=self.max_obs_chars)
                consecutive_errors = 0

            except Exception as exc:
                consecutive_errors += 1
                obs = f"ERROR - {type(exc).__name__}: {exc}"

                if consecutive_errors >= self.max_consecutive_errors:
                    scratchpad += f"\nObservation: {obs}\n"
                    fallback_answer = compute_answer(all_actions, self.source_df) if all_actions else None
                    return {
                        "actions": all_actions,
                        "answer": fallback_answer,
                        "error": f"Max consecutive errors ({self.max_consecutive_errors}) reached.",
                        "steps": step + 1,
                        "scratchpad": scratchpad,
                    }

            scratchpad += f"\nObservation: {obs}\n"

        fallback_answer = compute_answer(all_actions, self.source_df) if all_actions else None
        return {
            "actions": all_actions,
            "answer": fallback_answer,
            "error": f"Max steps ({self.max_steps}) reached without Final Answer.",
            "steps": self.max_steps,
            "scratchpad": scratchpad,
        }

    def _generate(self, prompt):
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SEQ_LENGTH_P2,
        ).to(self.model.device)

        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                stopping_criteria=self.stop_criteria,
            )

        new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True)

    def _parse_action(self, text):
        action_match = re.search(r"Action:\s*([A-Za-z_][A-Za-z0-9_]*)", text)
        input_match = re.search(
            r"Action Input:\s*(.*?)(?:\n(?:Observation:|Thought:|Action:|Final Answer:)|$)",
            text,
            re.DOTALL,
        )

        if not action_match or not input_match:
            raise ValueError(
                "Could not parse action. Expected 'Action: <name>' and 'Action Input: <json or legacy args>'."
            )

        action_name = action_match.group(1).strip()
        raw_input = input_match.group(1).strip()
        action_input = parse_action_input_flexible(action_name, raw_input)

        if not isinstance(action_input, dict):
            raise ValueError("Action Input must decode to a dictionary.")

        return action_name, action_input

    def _parse_final_answer(self, text):
        text = sanitize_json(text)

        try:
            return json.loads(text)
        except json.JSONDecodeError:
            pass

        try:
            return ast.literal_eval(text)
        except (ValueError, SyntaxError):
            pass

        match = re.search(r"\{.*\}|\[.*\]", text, re.DOTALL)
        if match:
            chunk = sanitize_json(match.group())
            try:
                return json.loads(chunk)
            except json.JSONDecodeError:
                pass
            try:
                return ast.literal_eval(chunk)
            except (ValueError, SyntaxError):
                pass

        num_match = re.search(r"-?[\d]+(?:\.\d+)?", text)
        if num_match:
            val = num_match.group()
            return float(val) if "." in val else int(val)

        return text


agent = ReActAgent(model_p2, tokenizer_p2, df, max_steps=5)

test_queries = [
    "What is the total revenue for 2022?",
    "Which city had the highest profit in 2021? Top 1",
    "What is the average revenue by city?",
]

for q in test_queries:
    print("=" * 80)
    print("Q:", q)
    result = agent.run(q)
    print("result:", result)
    print("tool-computed answer from actions:", compute_answer(result.get("actions", []), df))


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

ValueError: mount failed

In [ ]:
import gc
import glob as _glob
import os
import sys
import json
import re
import ast
import pandas as pd
import torch
from peft import PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    StoppingCriteria,
    StoppingCriteriaList,
)

DRIVE_DIR = "/content/drive/MyDrive/CS_F425_Project"
MODEL_NAME_P2 = "mistralai/Mistral-7B-v0.1"
ADAPTER_DIR_P2 = f"{DRIVE_DIR}/mistral-react-adapter_divyam_duplicate"
CKPT_DIR_P2 = f"{DRIVE_DIR}/mistral-react-qlora_divyam_duplicate"
MAX_SEQ_LENGTH_P2 = 384

sys.path.insert(0, DRIVE_DIR)
from tool_executor import ToolExecutor

df = pd.read_csv(f"{DRIVE_DIR}/sales_data.csv")

epoch_ckpts = _glob.glob(f"{CKPT_DIR_P2}/checkpoint-*")
timed_ckpts = _glob.glob(f"{ADAPTER_DIR_P2}/timed_ckpt_step_*")

print("epoch checkpoints:", epoch_ckpts[:5], "count =", len(epoch_ckpts))
print("timed checkpoints:", timed_ckpts[:5], "count =", len(timed_ckpts))

load_path = None
if epoch_ckpts:
    epoch_ckpts.sort(key=lambda p: int(p.rsplit("-", 1)[-1]))
    load_path = epoch_ckpts[-1]
elif timed_ckpts:
    timed_ckpts.sort(key=lambda p: int(p.rsplit("_", 1)[-1]))
    load_path = timed_ckpts[-1]
else:
    raise ValueError("No saved checkpoints found in either checkpoint-* or timed_ckpt_step_*")

print("Loading from:", load_path)

try:
    del model_p2
except:
    pass
gc.collect()
torch.cuda.empty_cache()

tokenizer_p2 = AutoTokenizer.from_pretrained(MODEL_NAME_P2, trust_remote_code=True)
if tokenizer_p2.pad_token is None:
    tokenizer_p2.pad_token = tokenizer_p2.eos_token
tokenizer_p2.padding_side = "right"

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb_config_p2 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

base_model_p2 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_P2,
    quantization_config=bnb_config_p2,
    device_map="auto",
    trust_remote_code=True,
)

model_p2 = PeftModel.from_pretrained(base_model_p2, load_path)
model_p2.eval()
model_p2.config.use_cache = True


In [ ]:
def _parse_numeric(val_str):
    return float(val_str) if "." in val_str else int(val_str)

def parse_agent_action(action_str):
    if action_str.startswith("filter_data"):
        m = re.search(r"column='([^']+)',\s*value=(-?[\d]+(?:\.\d+)?)", action_str)
        if m:
            return {"tool": "filter", "args": {"column": m.group(1), "op": "==", "value": _parse_numeric(m.group(2))}}
        m = re.search(r"column='([^']+)',\s*value='([^']+)'", action_str)
        if m:
            return {"tool": "filter", "args": {"column": m.group(1), "op": "==", "value": m.group(2)}}
    elif action_str.startswith("group_by"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "groupby", "args": {"column": m.group(1)}}
    elif action_str.startswith("aggregate_sum"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "sum"}}
    elif action_str.startswith("aggregate_mean"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "mean"}}
    elif action_str.startswith("aggregate_count"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "count"}}
    elif action_str.startswith("sort_by"):
        m = re.search(r"column='([^']+)',\s*order='([^']+)'", action_str)
        if m:
            return {"tool": "sort", "args": {"column": m.group(1), "ascending": m.group(2) != "desc"}}
    elif action_str.startswith("top_k"):
        m = re.search(r"k=(\d+)", action_str)
        if m:
            return {"tool": "topk", "args": {"k": int(m.group(1))}}
    return None

def execute_action_sequence(actions, source_df):
    parsed = []
    for action in actions:
        parsed_action = parse_agent_action(action)
        if parsed_action is None:
            raise ValueError(f"Could not parse action: {action}")
        parsed.append(parsed_action)
    return ToolExecutor(source_df.copy()).execute(parsed)

def clean_scalar(value):
    if hasattr(value, "item"):
        try:
            value = value.item()
        except Exception:
            pass
    if isinstance(value, bool):
        return bool(value)
    if isinstance(value, int):
        return int(value)
    if isinstance(value, float):
        return round(float(value), 4)
    return value

def result_to_python(actions, result_df):
    if result_df is None or (hasattr(result_df, "empty") and result_df.empty):
        return None
    if hasattr(result_df, "groups"):
        return None
    if getattr(result_df, "shape", None) == (1, 1):
        return clean_scalar(result_df.iloc[0, 0])
    if isinstance(result_df, pd.DataFrame):
        return [
            {str(key): clean_scalar(value) for key, value in row.items()}
            for row in result_df.to_dict(orient="records")
        ]
    return clean_scalar(result_df)

def compute_answer(actions, source_df):
    if not actions:
        return None
    try:
        result = execute_action_sequence(actions, source_df)
        return result_to_python(actions, result)
    except Exception:
        return None

REACT_SYSTEM_PROMPT = """You are a data analysis agent. You have access to the following tools to analyze a sales dataset:

Function Descriptions:
[
  {"name": "filter_data", "description": "Filter rows where column equals value", "parameters": {"column": "str", "value": "str or int"}},
  {"name": "group_by", "description": "Group the data by a column", "parameters": {"column": "str"}},
  {"name": "aggregate_sum", "description": "Sum a numeric column", "parameters": {"column": "str"}},
  {"name": "aggregate_mean", "description": "Average a numeric column", "parameters": {"column": "str"}},
  {"name": "aggregate_count", "description": "Count rows for a column", "parameters": {"column": "str"}},
  {"name": "sort_by", "description": "Sort by a column", "parameters": {"column": "str", "order": "asc or desc"}},
  {"name": "top_k", "description": "Select top k rows", "parameters": {"k": "int"}}
]

Schema: date (date), year (int), month (int), city (str), region (str), product (str), category (str), revenue (float), units_sold (int), cost (float), profit (float)

Use the Thought/Action/Action Input format. Wait for Observation after each action. End with Final Answer."""

REACT_PROMPT_TEMPLATE = """### System
{system}

### User Query
{question}

### Agent Scratchpad
{scratchpad}"""

def action_string_to_action_input(action_str):
    if action_str.startswith("filter_data"):
        m = re.search(r"column='([^']+)',\s*value=(-?[\d]+(?:\.\d+)?)", action_str)
        if m:
            return {"column": m.group(1), "value": _parse_numeric(m.group(2))}
        m = re.search(r"column='([^']+)',\s*value='([^']+)'", action_str)
        if m:
            return {"column": m.group(1), "value": m.group(2)}
    elif action_str.startswith("group_by"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"column": m.group(1)}
    elif action_str.startswith(("aggregate_sum", "aggregate_mean", "aggregate_count")):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"column": m.group(1)}
    elif action_str.startswith("sort_by"):
        m = re.search(r"column='([^']+)',\s*order='([^']+)'", action_str)
        if m:
            return {"column": m.group(1), "order": m.group(2)}
    elif action_str.startswith("top_k"):
        m = re.search(r"k=(\d+)", action_str)
        if m:
            return {"k": int(m.group(1))}
    raise ValueError(f"Unsupported action string: {action_str}")

def _quote_string(value):
    escaped = value.replace("\\", "\\\\").replace("'", "\\'")
    return f"'{escaped}'"

def _format_dsl_value(value):
    if isinstance(value, str):
        return _quote_string(value)
    if isinstance(value, bool):
        return "1" if value else "0"
    return str(value)

def action_input_to_action_string(action_name, action_input):
    if action_name == "filter_data":
        return f"filter_data(column={_quote_string(str(action_input['column']))}, value={_format_dsl_value(action_input['value'])})"
    if action_name == "group_by":
        return f"group_by(column={_quote_string(str(action_input['column']))})"
    if action_name in {"aggregate_sum", "aggregate_mean", "aggregate_count"}:
        return f"{action_name}(column={_quote_string(str(action_input['column']))})"
    if action_name == "sort_by":
        return f"sort_by(column={_quote_string(str(action_input['column']))}, order={_quote_string(str(action_input['order']))})"
    if action_name == "top_k":
        return f"top_k(k={int(action_input['k'])})"
    raise ValueError(f"Unknown action name: {action_name}")

def format_observation(result_df, max_chars=500):
    if result_df is None:
        return "No data returned."
    if hasattr(result_df, "groups") and not isinstance(result_df, pd.DataFrame):
        group_names = list(result_df.groups.keys())
        return f"Grouped result with {len(group_names)} groups. Sample groups: {[str(x) for x in group_names[:10]]}"
    if hasattr(result_df, "empty") and result_df.empty:
        return "Empty result."
    if getattr(result_df, "shape", None) == (1, 1):
        return str(result_df.iloc[0, 0])
    text = result_df.to_string(index=False) if isinstance(result_df, pd.DataFrame) else str(result_df)
    if len(text) > max_chars:
        text = text[:max_chars] + "..."
    return text

def sanitize_json(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^```json\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    text = re.sub(r",\s*([}\]])", r"\1", text)
    return text.strip()

def parse_json_like(text: str):
    cleaned = sanitize_json(text)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    try:
        return ast.literal_eval(cleaned)
    except (ValueError, SyntaxError):
        pass
    match = re.search(r"\{.*\}|\[.*\]", cleaned, re.DOTALL)
    if match:
        chunk = sanitize_json(match.group())
        try:
            return json.loads(chunk)
        except json.JSONDecodeError:
            pass
        try:
            return ast.literal_eval(chunk)
        except (ValueError, SyntaxError):
            pass
    raise ValueError(f"Could not parse structured input: {text}")

def parse_action_input_flexible(action_name: str, text: str):
    raw = text.strip()
    if not raw:
        raise ValueError("Empty Action Input.")
    try:
        parsed = parse_json_like(raw)
        if isinstance(parsed, dict):
            return parsed
    except Exception:
        pass
    try:
        parsed = action_string_to_action_input(f"{action_name}({raw})")
        if isinstance(parsed, dict):
            return parsed
    except Exception:
        pass
    raise ValueError(f"Could not parse Action Input for {action_name}: {raw}")

class ObservationStopCriteria(StoppingCriteria):
    def __init__(self, tokenizer, triggers=("Observation:", "Final Answer:")):
        self.tokenizer = tokenizer
        self.triggers = triggers
    def __call__(self, input_ids, scores, **kwargs):
        tail = self.tokenizer.decode(input_ids[0, -40:], skip_special_tokens=True)
        return any(trigger in tail for trigger in self.triggers)

class ReActAgent:
    def __init__(self, model, tokenizer, source_df, max_steps=5, max_obs_chars=500, max_consecutive_errors=2, max_new_tokens=256):
        self.model = model
        self.tokenizer = tokenizer
        self.source_df = source_df
        self.max_steps = max_steps
        self.max_obs_chars = max_obs_chars
        self.max_consecutive_errors = max_consecutive_errors
        self.max_new_tokens = max_new_tokens
        self.stop_criteria = StoppingCriteriaList([ObservationStopCriteria(tokenizer)])

    def run(self, question):
        self.model.eval()
        self.model.config.use_cache = True
        scratchpad = ""
        consecutive_errors = 0
        all_actions = []

        for step in range(self.max_steps):
            prompt = REACT_PROMPT_TEMPLATE.format(
                system=REACT_SYSTEM_PROMPT,
                question=question,
                scratchpad=scratchpad,
            )

            output_text = self._generate(prompt)
            scratchpad += output_text

            if "Final Answer:" in output_text:
                answer_text = output_text.split("Final Answer:", 1)[-1].strip()
                answer_text = answer_text.replace(self.tokenizer.eos_token or "", "").strip()
                return {
                    "actions": all_actions,
                    "answer": self._parse_final_answer(answer_text),
                    "steps": step + 1,
                    "scratchpad": scratchpad,
                }

            try:
                action_name, action_input = self._parse_action(output_text)
                action_str = action_input_to_action_string(action_name, action_input)

                if parse_agent_action(action_str) is None:
                    raise ValueError(f"Unknown action: {action_str}")

                all_actions.append(action_str)
                result = execute_action_sequence(all_actions, self.source_df)
                obs = format_observation(result, max_chars=self.max_obs_chars)
                consecutive_errors = 0

            except Exception as exc:
                consecutive_errors += 1
                obs = f"ERROR - {type(exc).__name__}: {exc}"
                if consecutive_errors >= self.max_consecutive_errors:
                    scratchpad += f"\nObservation: {obs}\n"
                    fallback_answer = compute_answer(all_actions, self.source_df) if all_actions else None
                    return {
                        "actions": all_actions,
                        "answer": fallback_answer,
                        "error": f"Max consecutive errors ({self.max_consecutive_errors}) reached.",
                        "steps": step + 1,
                        "scratchpad": scratchpad,
                    }

            scratchpad += f"\nObservation: {obs}\n"

        fallback_answer = compute_answer(all_actions, self.source_df) if all_actions else None
        return {
            "actions": all_actions,
            "answer": fallback_answer,
            "error": f"Max steps ({self.max_steps}) reached without Final Answer.",
            "steps": self.max_steps,
            "scratchpad": scratchpad,
        }

    def _generate(self, prompt):
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SEQ_LENGTH_P2,
        ).to(self.model.device)

        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                stopping_criteria=self.stop_criteria,
            )

        new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True)

    def _parse_action(self, text):
        action_match = re.search(r"Action:\s*([A-Za-z_][A-Za-z0-9_]*)", text)
        input_match = re.search(
            r"Action Input:\s*(.*?)(?:\n(?:Observation:|Thought:|Action:|Final Answer:)|$)",
            text,
            re.DOTALL,
        )
        if not action_match or not input_match:
            raise ValueError("Could not parse action.")

        action_name = action_match.group(1).strip()
        raw_input = input_match.group(1).strip()
        action_input = parse_action_input_flexible(action_name, raw_input)

        if not isinstance(action_input, dict):
            raise ValueError("Action Input must decode to a dictionary.")

        return action_name, action_input

    def _parse_final_answer(self, text):
        text = sanitize_json(text)
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            pass
        try:
            return ast.literal_eval(text)
        except (ValueError, SyntaxError):
            pass
        match = re.search(r"\{.*\}|\[.*\]", text, re.DOTALL)
        if match:
            chunk = sanitize_json(match.group())
            try:
                return json.loads(chunk)
            except json.JSONDecodeError:
                pass
            try:
                return ast.literal_eval(chunk)
            except (ValueError, SyntaxError):
                pass
        num_match = re.search(r"-?[\d]+(?:\.\d+)?", text)
        if num_match:
            val = num_match.group()
            return float(val) if "." in val else int(val)
        return text


agent = ReActAgent(model_p2, tokenizer_p2, df, max_steps=5)

test_queries = [
    "What is the total revenue for 2022?",
    "Which city had the highest profit in 2021? Top 1",
    "What is the average revenue by city?",
]

for q in test_queries:
    print("=" * 80)
    print("Q:", q)
    result = agent.run(q)
    print("result:", result)
    print("tool-computed answer from actions:", compute_answer(result.get("actions", []), df))
